# HSCredit 建模参考代码

本 Notebook 使用仓库自带的真实放款样例 `hscredit_yyp.xlsx`，演示一条可重复执行的评分卡建模链路：数据校验、分层划分、最优分箱与 WOE、ScoreCard 训练、效果评估以及模型持久化。

## 1. 目标与约定

- 建模特征：`衡枢鉴真分老客版`、`近六个月非银多头机构数`、`青云24`
- 目标变量：`FPD`，其中 1 表示坏样本
- 划分方式：按目标变量分层抽样，70% 训练、30% 测试
- 分数方向：信用分越高，预测坏样本概率越低

In [ ]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def locate_examples_dir() -> Path:
    """兼容从仓库根目录或 examples 目录启动 Notebook。"""
    candidates = (Path.cwd(), Path.cwd() / 'examples')
    for candidate in candidates:
        if (candidate / 'hscredit_yyp.xlsx').is_file():
            return candidate.resolve()
    raise FileNotFoundError('未找到 examples/hscredit_yyp.xlsx，请从仓库根目录或 examples 目录运行。')


EXAMPLES_DIR = locate_examples_dir()
PROJECT_ROOT = EXAMPLES_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import ks
from hscredit.core.models import ScoreCard

print(f'项目目录：{PROJECT_ROOT}')
print(f'样例数据：{EXAMPLES_DIR / "hscredit_yyp.xlsx"}')

## 2. 加载并校验数据

先验证字段、样本数、缺失情况与标签取值，避免把输入问题带入建模环节。

In [ ]:
FEATURES = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24']
TARGET = 'FPD'
EXPECTED_ROWS = 970

raw_data = pd.read_excel(EXAMPLES_DIR / 'hscredit_yyp.xlsx')
missing_columns = sorted(set(FEATURES + [TARGET]) - set(raw_data.columns))
assert not missing_columns, f'样例数据缺少字段：{missing_columns}'

model_data = raw_data[FEATURES + [TARGET]].copy()
assert len(model_data) == EXPECTED_ROWS, f'样例数据行数应为 {EXPECTED_ROWS}，实际为 {len(model_data)}'
assert model_data[FEATURES].isna().sum().sum() == 0, '建模特征存在缺失值'
assert set(model_data[TARGET].unique()) == {0, 1}, 'FPD 必须是同时包含 0 和 1 的二分类标签'

print(f'样本总数：{len(model_data):,}')
print(f'建模特征数：{len(FEATURES)}')
print(f'坏样本数：{int(model_data[TARGET].sum()):,}')
print(f'总体坏样本率：{model_data[TARGET].mean():.2%}')
display(model_data.head())

## 3. 分层划分训练集与测试集

固定随机种子并按 `FPD` 分层，使执行结果可重复，并让训练集、测试集的坏样本率保持接近。

In [ ]:
X = model_data[FEATURES]
y = model_data[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

assert len(X_train) + len(X_test) == len(model_data), '训练集与测试集样本数不守恒'
assert X_train.index.intersection(X_test.index).empty, '训练集与测试集存在样本重叠'
assert y_train.nunique() == 2 and y_test.nunique() == 2, '训练集或测试集缺少某一类标签'

split_summary = pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    '样本数': [len(X_train), len(X_test)],
    '坏样本数': [int(y_train.sum()), int(y_test.sum())],
    '坏样本率': [y_train.mean(), y_test.mean()],
})
display(split_summary.style.format({'坏样本率': '{:.2%}'}))

## 4. 最优分箱与 WOE 转换

分箱器只在训练集上拟合，再用同一套规则转换测试集，避免数据泄漏。

In [ ]:
binner = OptimalBinning(method='target_bad_rate', max_n_bins=5)
binner.fit(X_train, y_train)

X_train_woe = binner.transform(X_train, metric='woe')
X_test_woe = binner.transform(X_test, metric='woe')

assert list(X_train_woe.columns) == FEATURES, 'WOE 转换后的字段顺序发生变化'
assert X_train_woe.shape == X_train.shape and X_test_woe.shape == X_test.shape, 'WOE 转换前后维度不一致'
assert np.isfinite(X_train_woe.to_numpy(dtype=float)).all(), '训练集 WOE 中存在非有限值'
assert np.isfinite(X_test_woe.to_numpy(dtype=float)).all(), '测试集 WOE 中存在非有限值'

print(f'训练集 WOE 维度：{X_train_woe.shape}')
print(f'测试集 WOE 维度：{X_test_woe.shape}')
display(X_train_woe.head())

In [ ]:
first_feature_bin_table = binner.get_bin_table(FEATURES[0])
assert int(first_feature_bin_table['样本总数'].sum()) == len(X_train), '分箱表样本数与训练集不一致'
print(f'特征“{FEATURES[0]}”的训练集分箱结果：')
display(first_feature_bin_table)

## 5. 训练 ScoreCard

`ScoreCard` 接收训练集 WOE，并保留上一步分箱器；预测时可以直接传入原始特征。

In [ ]:
scorecard = ScoreCard(
    binner=binner,
    base_score=600,
    pdo=50,
    rate=2,
    base_odds=20,
    direction='descending',
)
scorecard.fit(X_train_woe, y_train, input_type='woe')

train_probability = scorecard.predict_proba(X_train)[:, 1]
test_probability = scorecard.predict_proba(X_test)[:, 1]
train_score = scorecard.predict(X_train, input_type='raw')
test_score = scorecard.predict(X_test, input_type='raw')

for name, probability, score in (
    ('训练集', train_probability, train_score),
    ('测试集', test_probability, test_score),
):
    assert np.isfinite(probability).all() and np.isfinite(score).all(), f'{name}预测存在非有限值'
    assert ((probability >= 0) & (probability <= 1)).all(), f'{name}预测概率超出 [0, 1]'

print(f'测试集评分范围：{test_score.min():.2f} ~ {test_score.max():.2f}')
print(f'测试集平均预测坏样本概率：{test_probability.mean():.2%}')

## 6. 评估并做方向检查

AUC 与 KS 检查模型排序能力；信用分和坏样本概率应呈负相关。这里的阈值只用于验证演示链路有效，不代表生产准入标准。

In [ ]:
evaluation = pd.DataFrame([
    {
        '数据集': '训练集',
        '样本数': len(y_train),
        'AUC': roc_auc_score(y_train, train_probability),
        'KS': ks(y_train, train_probability),
    },
    {
        '数据集': '测试集',
        '样本数': len(y_test),
        'AUC': roc_auc_score(y_test, test_probability),
        'KS': ks(y_test, test_probability),
    },
])

score_probability_correlation = float(np.corrcoef(test_score, test_probability)[0, 1])
assert evaluation['AUC'].between(0.5, 1.0).all(), 'AUC 未达到有效排序的最低检查线'
assert evaluation['KS'].between(0.0, 1.0).all(), 'KS 超出合理范围'
assert score_probability_correlation < 0, '信用分与坏样本概率方向不一致'

display(evaluation.style.format({'AUC': '{:.4f}', 'KS': '{:.4f}'}))
print(f'测试集信用分与坏样本概率相关系数：{score_probability_correlation:.4f}')

In [ ]:
scorecard_points = scorecard.scorecard_points()
assert not scorecard_points.empty, '评分卡分值表为空'
print(f'评分卡规则行数：{len(scorecard_points)}')
display(scorecard_points)

## 7. 模型持久化一致性

使用 `ScoreCard.save_pickle` / `ScoreCard.load_pickle` 保存并回载模型。文件放入系统临时目录，验证结束后自动清理，不污染仓库。

In [ ]:
with TemporaryDirectory(prefix='hscredit_scorecard_') as temporary_directory:
    model_path = Path(temporary_directory) / 'scorecard.pkl'
    scorecard.save_pickle(str(model_path))
    loaded_scorecard = ScoreCard.load_pickle(str(model_path))
    loaded_test_score = loaded_scorecard.predict(X_test, input_type='raw')

    max_score_difference = float(np.max(np.abs(test_score - loaded_test_score)))
    assert model_path.is_file(), '模型文件保存失败'
    assert max_score_difference < 1e-12, '模型回载前后评分不一致'

    print(f'临时模型文件：{model_path.name}')
    print(f'回载前后评分最大差异：{max_score_difference:.12f}')

print('临时文件已自动清理。')

## 8. 下一步

本 Notebook 聚焦可复用的基础建模链路。若要生成包含多数据集、分箱、稳定性、金额与逾期口径的 Excel 模型报告，请继续运行：

- `11_report.ipynb`：报告 API 与内容专项演示
- `12_complete_workflow.ipynb`：完整工作流及报告落盘演示

生产使用前还应补充时间外验证、稳定性监控、变量治理与业务阈值评审。